# Pre Call Metrics

调用AI前的认知状态。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.insert(0, '../../modules/semantic_toolbox')

from semdistance.distance import dis_sentences

In [ ]:
# 输入文件路径
data_dir = Path('../../data/pickle')
timeline_file = data_dir / 'timeline.pkl'
ai_events_file = data_dir / 'ai_events.pkl'
user_msgs_file = data_dir / 'user_msgs.pkl'

# 输出文件路径
output_dir = Path('../../data/analysis/pre_call')
pre_call_metrics_file = output_dir / 'pre_call_metrics.csv'
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 加载数据
timeline = pd.read_pickle(timeline_file)
ai_events = pd.read_pickle(ai_events_file)
user_msgs = pd.read_pickle(user_msgs_file)

In [ ]:
def compute_item_semantic_metrics(item_name, prev_messages, dis_func):
    distances = []

    for msg in prev_messages:
        content = msg['content']
        if pd.isna(content) or content is None:
            distances.append(np.nan)
            continue

        try:
            distances.append(dis_func(content, item_name, 1))
        except Exception as e:
            print(f'语义距离计算出错：{content} vs {item_name}，错误：{e}')
            distances.append(np.nan)

    valid_distances = [value for value in distances if pd.notna(value)]
    semantic_distance_mean = np.mean(valid_distances) if valid_distances else np.nan
    if len(valid_distances) >= 2:
        semantic_distance_change = valid_distances[-1] - valid_distances[0]
    else:
        semantic_distance_change = np.nan

    while len(distances) < 3:
        distances.append(np.nan)

    return {
        'semantic_distance_1': distances[0],
        'semantic_distance_2': distances[1],
        'semantic_distance_3': distances[2],
        'semantic_distance_mean': semantic_distance_mean,
        'semantic_distance_change': semantic_distance_change,
        'semantic_distance_valid_count': len(valid_distances),
    }

In [ ]:
# 对每个AI点击，计算调用前指标
pre_call_metrics = []

for idx, row in ai_events.iterrows():
    pid = row['participant_id']
    item = row['item_name']
    trial = row['trial_index']
    click_time = row['click_time']
    
    # 找到本次点击之前最后一次用户消息
    prev_user = user_msgs[
        (user_msgs['participant_id'] == pid) &
        (user_msgs['item_name'] == item) &
        (user_msgs['trial_index'] == trial) &
        (user_msgs['time'] < click_time)
    ].sort_values('time').tail(1)
    
    # 1. 调用前思考时间
    if len(prev_user) == 0:
        # 如果之前没有用户消息，思考时长为点击时间本身
        pre_think_time = click_time
        last_user_time = 0
        last_user_content = None
    else:
        last_user_time = prev_user.iloc[0]['time']
        pre_think_time = click_time - last_user_time
        last_user_content = prev_user.iloc[0]['content']
    
    # 计算平均思考时长（该被试-物品的所有相邻用户消息间隔平均值）
    all_user = user_msgs[
        (user_msgs['participant_id'] == pid) &
        (user_msgs['item_name'] == item) &
        (user_msgs['trial_index'] == trial)
    ].sort_values('time')
    if len(all_user) >= 2:
        user_intervals = np.diff(all_user['time'].values)
        avg_think_time = np.mean(user_intervals)
    else:
        avg_think_time = np.nan
    
    # 2. 相对停顿时间
    if avg_think_time and avg_think_time > 0:
        relative_pause = pre_think_time / avg_think_time
    else:
        relative_pause = np.nan
    
    # 调用前语义距离：需要最近3个想法的语义距离，需要外部语义数据
    prev_3 = user_msgs[
        (user_msgs['participant_id'] == pid) &
        (user_msgs['item_name'] == item) &
        (user_msgs['trial_index'] == trial) &
        (user_msgs['time'] < click_time)
    ].sort_values('time').tail(3)
    
    semantic_metrics = compute_item_semantic_metrics(item, prev_3.to_dict('records'), dis_sentences)
    
    pre_call_metrics.append({
        'participant_id': pid,
        'item_name': item,
        'trial_index': trial,
        'click_index': row['click_index'],
        'click_time': click_time,
        'pre_think_time': pre_think_time,
        'avg_think_time': avg_think_time,
        'relative_pause': relative_pause,
        'last_user_time': last_user_time,
        'last_user_content': last_user_content,
        **semantic_metrics
    })

In [ ]:
pre_call_df = pd.DataFrame(pre_call_metrics)
pre_call_df = pre_call_df.sort_values(['participant_id', 'trial_index'])
pre_call_df.to_csv(pre_call_metrics_file, index=False)